## attribute override

This notebook introduces `attribute :>>` override; after running it you can express named variants of a design by overriding inherited attribute values.

The previous notebook declared that the toaster must complete a cycle in at most 180 seconds. Before checking whether the design meets that requirement, we need to state the operating conditions we are designing for. This notebook introduces `attribute :>>` override: a part usage can redeclare an inherited attribute with a specific value, encoding the assumption being evaluated.

In [ ]:
import opensysml
from toaster.report import format_diagnostics

source = """
package ToasterDemo {
    private import ScalarValues::*;

    abstract part def ToastingSystem {
        doc /* Transform bread into toast acceptable to its user. */
    }

    part def Heater {
        attribute power : Real default = 800.0;
    }

    part def HeatingSystem :> ToastingSystem;
    part def ControlSystem :> ToastingSystem;

    part def Toaster {
        attribute cycleTime : Real default = 120.0;
        part heating : HeatingSystem;
        part control : ControlSystem;
    }

    requirement def TimelyToast {
        subject toaster : Toaster;
        require constraint { toaster.cycleTime <= 180.0 }
    }

    part nominal : Toaster;
    part slow : Toaster {
        attribute :>> cycleTime = 200.0;
    }
}
"""

conn = opensysml.connect(version="v0.9.0")
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"

In [ ]:
# Negative control: :>> can only override an attribute that already
# exists in the inherited chain. Overriding a non-existent name fails.
bad_source = """
package Bad {
    private import ScalarValues::*;
    part def Toaster { attribute cycleTime : Real default = 120.0; }
    part slow : Toaster {
        attribute :>> nonExistent = 200.0;
    }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok
print("Expected error:", bad.diagnostics[0].message)

In [ ]:
nominal = model.find("ToasterDemo::nominal")
slow = model.find("ToasterDemo::slow")
assert nominal is not None
assert slow is not None

print("nominal:", nominal.id, "| kind:", nominal.kind)
print("slow   :", slow.id, "| kind:", slow.kind)

slow_attrs = slow.attributes()
print(f"slow overridden attributes ({len(slow_attrs)}):")
for a in slow_attrs:
    print(f"  {a.id}")
conn.close()

`part slow : Toaster { attribute :>> cycleTime = 200.0; }` is the A-F override; OpenSysML resolves the redeclaration against the inherited attribute from `Toaster` (O-S); `slow.attributes()` returns the overridden symbol (E).

Try the chapter exercise in `exercises/ch02/exercise.ipynb`: create a `weakBrew` variant of your `CoffeeMaker` with a lower `brewTemp` and confirm the override loads.